In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yasa
import psutil
from psutil._common import bytes2human

from pathlib import Path
from scipy.io import loadmat
from mne.filter import resample
from itertools import chain
from tqdm import tqdm
from sys import getsizeof

from functions import get_sequences
from functions import phasic_rem_v3
from functions import create_name_cbd, create_name_rgs, create_name_os
from runtime_logger import logger_setup

CBD_DIR = "/home/nero/datasets/CBD/"
RGS_DIR = "/home/nero/datasets/RGS14/"
OS_DIR = "/home/nero/datasets/OSbasic/"

OUTPUT_DIR  = "/home/nero/datasets/preprocesssed/"
CBD_OVERVIEW_PATH = "overview.csv"

overview_df = pd.read_csv(CBD_OVERVIEW_PATH)

fs_cbd = 2500
fs_os = 2500
fs_rgs = 1000

targetFs = 500
n_down_cbd = fs_cbd/targetFs
n_down_rgs = fs_rgs/targetFs
n_down_os = fs_os/targetFs

def get_sequences(x, ibreak=1):
    """
    Identifies contiguous sequences.

    Parameters:
    x (np.ndarray): 1D time series.
    ibreak (int): A threshold value for determining breaks between sequences (default is 1).

    Returns:
    list of tuples: Each tuple contains the start and end integer of each contiguous sequence.
    """
    if len(x) == 0:
        return []

    diff = np.diff(x)
    breaks = np.where(diff > ibreak)[0]

    # Append the last index to handle the end of the array
    breaks = np.append(breaks, len(x) - 1)
    
    sequences = []
    start_idx = 0
    
    for break_idx in breaks:
        end_idx = break_idx
        sequences.append((x[start_idx], x[end_idx]))
        start_idx = end_idx + 1
    
    return sequences

def get_segments(idx, signal):
    """
    Extracts segments of the signal between specified start and end time indices.

    Parameters:
    idx (list of tuples): Each tuple contains (start_time, end_time).
    signal (np.ndarray): The signal from which to extract segments.

    Returns:
    list of np.ndarray: Each element is a segment of the signal corresponding to the given time ranges.
    """
    segments = []
    for (start_time, end_time) in idx:
        if end_time > len(signal):
            end_time = len(signal) - 1
        segment = signal[start_time:end_time]
        segments.append(segment)
    
    return segments

def _detect_troughs(signal, thr):
    lidx  = np.where(signal[0:-2] > signal[1:-1])[0]
    ridx  = np.where(signal[1:-1] <= signal[2:])[0]
    thidx = np.where(signal[1:-1] < thr)[0]
    sidx = np.intersect1d(lidx, np.intersect1d(ridx, thidx))+1
    return sidx

In [2]:
pattern = r"[\w-]+posttrial[\w-]+"
mapped = {}

for root, dirs, fils in os.walk(CBD_DIR):
    for dir in dirs:
        # Check if the directory is a post trial directory
        if re.match(pattern, dir, flags=re.IGNORECASE):
            dir = Path(os.path.join(root, dir))
            HPC_file = next(dir.glob("*HPC*continuous*"))
            states = next(dir.glob('*-states*'))
            mapped[str(states)] = str(HPC_file)

print("Number of recordings: ", len(mapped))

Number of recordings:  170


In [5]:
keys = iter(mapped.keys())
state = next(keys)
hpc = mapped[state]
title = create_name_cbd(hpc, overview_df)

# Load the LFP data
lfpHPC = loadmat(hpc)['HPC']
lfpHPC = lfpHPC.flatten()

# Load the states
hypno = loadmat(state)['states']
hypno = hypno.flatten()

# Downsample to 500 Hz
data_resample = resample(lfpHPC, down=n_down_cbd, method='fft', verbose="INFO")

# Remove artifacts
art_std, _ = yasa.art_detect(data_resample, targetFs , window=1, method='std', threshold=4)
art_up = yasa.hypno_upsample_to_data(art_std, 1, data_resample, targetFs)
data_resample[art_up] = 0
data_resample = data_resample - data_resample.mean()

rem_seq = get_sequences(np.where(hypno == 5)[0])
# Another representation: matrix of 2 columns and n rows (n number of rem epochs), first row is the start and second is for end idx.

rem_seq = [(start, end) for start, end in rem_seq]

# get REM segments
rem_idx = [(start * targetFs, (end+1) * targetFs) for start, end in rem_seq]
rem_epochs = get_segments(rem_idx, data_resample)

# Combine the REM indices with the corresponding downsampled segments
rem = {seq:seg for seq, seg in zip(rem_seq, rem_epochs)}

28-May-24 12:54:24 | WARNING | Hypnogram is SHORTER than data by 0.14 seconds. Padding hypnogram with last value to match data.size.


In [6]:
rem

{(1331,
  1485): array([-160.52095319,  -89.83372004,  -95.24662895, ..., -233.89630087,
        -222.18129903, -129.92739327]),
 (2428,
  2475): array([  98.89448376,    6.40722731,  -14.7707277 , ..., -375.02657014,
        -302.29221818, -248.92148926])}

In [7]:
from neurodsp.filt import filter_signal
from scipy.signal import hilbert

w1 = 5.0
w2 = 12.0
nfilt = 11
fs = targetFs
thr_dur = 900

tridx_list = []
trdiff_list = []
rem_eeg = np.array([])
eeg_seq = {}
sdiff_seq = {}
tridx_seq = {}
filt = np.ones((nfilt,))
filt = filt / filt.sum()

for idx in rem:
    start, end = idx

    epoch = rem[idx]
    epoch = filter_signal(epoch, fs, 'bandpass', (w1,w2), remove_edges=False)
    epoch = hilbert(epoch)

    inst_phase = np.angle(epoch)
    inst_amp = np.abs(epoch)

    # trough indices
    tridx = _detect_troughs(inst_phase, -3)
    
    # alternative version:
    #tridx = np.where(np.diff(np.sign(np.diff(eegh))))[0]+1
    
    # differences between troughs
    trdiff = np.diff(tridx)
       
    # smoothed trough differences
    sdiff_seq[idx] = np.convolve(trdiff, filt, 'same')

    # dict of trough differences for each REM period
    tridx_seq[idx] = tridx
        
    eeg_seq[idx] = inst_amp

    # differences between troughs
    trdiff_list += list(trdiff)
    
    # amplitude of the entire REM sleep
    rem_eeg = np.concatenate((rem_eeg, inst_amp)) 
    
trdiff = np.array(trdiff_list)
trdiff_sm = np.convolve(trdiff, filt, 'same')

# potential candidates for phasic REM:
# the smoothed difference between troughs is less than
# the 10th percentile:
thr1 = np.percentile(trdiff_sm, 10)
# the minimum difference in the candidate phREM is less than
# the 5th percentile
thr2 = np.percentile(trdiff_sm, 5)
# the peak amplitude is larger than the mean of the amplitude
# of the REM EEG.
thr3 = rem_eeg.mean()

In [182]:
print(thr1, thr2, thr3)

59.18181818181819 55.45454545454546 152.79370736721773


In [10]:
phrem = {}
for idx in tridx_seq:
    rem_start, rem_end = idx
    offset = rem_start * fs
    # trough indices
    tridx = tridx_seq[idx]
    # smoothed trough interval
    sdiff = sdiff_seq[idx]
    # ampplitude of the REM epoch
    eegh = eeg_seq[idx]
    cand_idx = np.where(sdiff <= thr1)[0]
    cand = get_sequences(cand_idx)
    
    for start, end in cand:
        dur = ( (tridx[end]-tridx[start]+1)/fs ) * 1000
        if dur > thr_dur and np.min(sdiff[start:end]) < thr2 and np.mean(eegh[tridx[start]:tridx[end]+1]) > thr3:
            a = tridx[start]   + offset
            b = tridx[end]  + offset
            
            if b > (rem_end * fs):
                b = rem_end*fs
                
            ph_idx = (a,b)
            if idx in phrem:
                phrem[idx].append(ph_idx)
            else:
                phrem[idx] = [ph_idx]

In [184]:
cand

[(0, 4), (16, 37), (154, 157), (358, 361)]

In [185]:
phrem

{(1331, 1485): [(689991, 690884), (705484, 707231), (729634, 730948)],
 (2428, 2475): [(1215016, 1216201)]}

In [186]:
[(end-start)/500 for (start, end) in list(phrem.values())[0]]

[1.786, 3.494, 2.628]

In [187]:
[(end-start)/500 for (start, end) in list(phrem.values())[1]]

[2.37]

In [117]:
def get_sequences(idx, ibreak=1) :  
    """
    get_sequences(idx, ibreak=1)
    idx     -    np.vector of indices
    @RETURN:
    seq     -    list of np.vectors
    """
    diff = idx[1:] - idx[0:-1]
    breaks = np.nonzero(diff>ibreak)[0]
    breaks = np.append(breaks, len(idx)-1)
    
    seq = []    
    iold = 0
    for i in breaks:
        r = list(range(iold, i+1))
        seq.append(idx[r])
        iold = i+1
    
    return seq

In [118]:
keys = iter(mapped.keys())
state = next(keys)
hpc = mapped[state]
title = create_name_cbd(hpc, overview_df)

# Load the LFP data
lfpHPC = loadmat(hpc)['HPC']
lfpHPC = lfpHPC.flatten()

# Load the states
hypno = loadmat(state)['states']
hypno = hypno.flatten()

data_resample = resample(lfpHPC, down=n_down_cbd, method='fft', verbose="INFO")
art_std, _ = yasa.art_detect(data_resample, targetFs , window=1, method='std', threshold=4, verbose='info')
art_up = yasa.hypno_upsample_to_data(art_std, 1, data_resample, targetFs)
data_resample[art_up] = 0

seq = get_sequences(np.where(hypno==5)[0])

25-May-24 21:02:56 | INFO | Number of samples in data = 1350571
25-May-24 21:02:56 | INFO | Sampling frequency = 500.00 Hz
25-May-24 21:02:56 | INFO | Data duration = 2701.14 seconds
25-May-24 21:02:56 | INFO | Trimmed standard deviation of CHAN000 = 218.1139 uV
25-May-24 21:02:56 | INFO | Peak-to-peak amplitude of CHAN000 = 12915.2322 uV
25-May-24 21:02:56 | INFO | Number of channels in data = 1
25-May-24 21:02:56 | INFO | Number of samples in data = 1350571
25-May-24 21:02:56 | INFO | Sampling frequency = 500.00 Hz
25-May-24 21:02:56 | INFO | Data duration = 2701.14 seconds
25-May-24 21:02:56 | INFO | Number of epochs = 2701
25-May-24 21:02:56 | INFO | Artifact window = 1.00 seconds
25-May-24 21:02:56 | INFO | Method = std
25-May-24 21:02:56 | INFO | Threshold = 4.00 standard deviations
25-May-24 21:02:56 | INFO | TOTAL: 48 / 2701 epochs rejected (1.78%)
25-May-24 21:02:56 | WARNING | Hypnogram is SHORTER than data by 0.14 seconds. Padding hypnogram with last value to match data.size

In [119]:
EEG = data_resample-data_resample.mean()
neeg = EEG.shape[0]

sr = targetFs
nbin = sr
        
w1 = 5.0
w2 = 12.0

filt = np.ones((nfilt,))
filt = filt / filt.sum()

trdiff_list = []
tridx_list = []
rem_eeg = np.array([])
eeg_seq = {}
sdiff_seq = {}
tridx_seq = {}
        
# Collect for each REM sequence the smoothed inter-trough intervals
# and EEG amplitudes as well as the indices of the troughs.
seq = [s for s in seq if len(s)>=min_dur]
for s in seq:
    ta = s[0]*nbin
    #tb = s[-1]*(nbin+1)
    tb = (s[-1]+1)*nbin
    tb = np.min((tb, neeg))
            
    eeg_idx = np.arange(ta, tb) # this the whole REM epoch       
    eeg = EEG[eeg_idx]

    print(eeg.shape)
    print(eeg)

    eegh =  filter_signal(eeg, sr, 'bandpass',(w1,w2), remove_edges=False)
    res = hilbert(eegh)
    instantaneous_phase = np.angle(res)
    amp = np.abs(res)

    # trough indices
    tridx = _detect_troughs(instantaneous_phase, -3)
    # Alternative that does not seems to work that well:        
    #tridx = np.where(np.diff(np.sign(np.diff(eegh))))[0]+1
    
    # differences between troughs
    trdiff = np.diff(tridx)
        
    # smoothed trough differences
    sdiff_seq[s[0]] = np.convolve(trdiff, filt, 'same')
    # dict of trough differences for each REM period
    tridx_seq[s[0]] = tridx

    eeg_seq[s[0]] = amp
        
# collect again smoothed inter-trough differences and amplitude;
# but this time concat the data to one long vector each (@trdiff_sm and rem_eeg)
for s in seq:
    ta = s[0]*nbin
    tb = (s[-1]+1)*nbin
    tb = np.min((tb, neeg))
    eeg_idx = np.arange(ta, tb)
    eeg = EEG[eeg_idx]            
    if len(eeg)*(1/sr) <= min_dur:
        continue
    
    eegh = filter_signal(eeg, sr, 'bandpass',(w1,w2), remove_edges=False)
    res = hilbert(eegh)
    instantaneous_phase = np.angle(res)
    amp = np.abs(res)

    # trough indices
    tridx = _detect_troughs(instantaneous_phase, -3)
    # alternative version:
    #tridx = np.where(np.diff(np.sign(np.diff(eegh))))[0]+1
    # differences between troughs
    tridx_list.append(tridx+ta)
    trdiff = np.diff(tridx)
    trdiff_list += list(trdiff)

    rem_eeg = np.concatenate((rem_eeg, amp)) 
    
trdiff = np.array(trdiff_list)
trdiff_sm = np.convolve(trdiff, filt, 'same')

# potential candidates for phasic REM:
# the smoothed difference between troughs is less than
# the 10th percentile:
thr1 = np.percentile(trdiff_sm, 10)
# the minimum difference in the candidate phREM is less than
# the 5th percentile
thr2 = np.percentile(trdiff_sm, 5)
# the peak amplitude is larger than the mean of the amplitude
# of the REM EEG.
thr3 = rem_eeg.mean()

phrem = {}
for si in tridx_seq:
    offset = nbin*si
    
    tridx = tridx_seq[si]
    sdiff = sdiff_seq[si]
    eegh = eeg_seq[si]
    
    idx = np.where(sdiff <= thr1)[0]
    cand = get_sequences(idx)

    #thr4 = np.mean(eegh)    
    for q in cand:
        dur = ( (tridx[q[-1]]-tridx[q[0]]+1)/sr ) * 1000
        #if 16250 > si*nbin * (1/sr) > 16100:
        #    print((tridx[q[0]]+si*nbin) * (1/sr))
        if dur > thr_dur and np.min(sdiff[q]) < thr2 and np.mean(eegh[tridx[q[0]]:tridx[q[-1]]+1]) > thr3:
            
            a = tridx[q[0]]   + offset
            b = tridx[q[-1]]  + offset
            idx = (a,b)

            if si in phrem:
                phrem[si].append(idx)
            else:
                phrem[si] = [idx]

(77500,)
[-160.52095319  -89.83372004  -95.24662895 ... -233.89630087 -222.18129903
 -129.92739327]
(24000,)
[  98.89448376    6.40722731  -14.7707277  ... -375.02657014 -302.29221818
 -248.92148926]


In [95]:
print(thr1, thr2, thr3)

59.18181818181819 55.45454545454546 152.79370736721773


In [96]:
phrem

{1331: [(689991, 690883), (705484, 707230), (729634, 730947)],
 2428: [(1215016, 1216200)]}

In [99]:
cand

[array([0, 1, 2, 3, 4]),
 array([16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32,
        33, 34, 35, 36, 37]),
 array([154, 155, 156, 157]),
 array([358, 359, 360, 361])]